# SciGeoGuard-X — Publication Experiments v3
## Real-pixel verification, unique SilentGeoBench cases, blind compositions, and publication exports

This notebook is the next-stage **paper experiment notebook**.

It directly fixes the problems found in v2:

1. **Sentinel-2:** no fixed centre window. The notebook searches for a real low-cloud scene and automatically finds a window with usable pixels.
2. **WorldCover:** categorical data are first read into a numeric array, then explicitly interpolated with `scipy.ndimage.zoom`. GDAL/Rasterio cannot silently fall back to nearest-neighbour.
3. **Benchmark diversity:** cases are generated until their **scientific configurations are unique**.
4. **Evaluation leakage:** the verifier never receives `family`, `split`, `tier`, or `expected`; it sees only artifacts, operations, and scientific intent.
5. **Blind compositions:** a third split combines operations in ways not used by the single-family development generators.
6. **Publication package:** results, figures, benchmark JSONL/CSV, review sheets, source provenance, and a ZIP are exported automatically.

> Generated cases are machine-created stress tests. Do not call them expert-reviewed until qualified reviewers actually annotate them.

## 0. Execution modes

Set `LIVE_DATA = True` in Colab for the real EO experiments.

The notebook is saved and validated with `LIVE_DATA = False`, so the formal verifier, benchmark generation, evaluation, figures, and exports run even without network access.

In [ ]:
LIVE_DATA = False
SEED = 20261002

from dataclasses import dataclass, field, asdict, replace
from pathlib import Path
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional
import hashlib, json, math, random, itertools, shutil, warnings

import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

RUN_DIR = Path("/content/scigeoguard_v3") if Path("/content").exists() else Path("/mnt/data/scigeoguard_v3")
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v3")
print("LIVE_DATA =", LIVE_DATA)
print("RUN_DIR   =", RUN_DIR)

# Part I — Source registry and real-data provenance

The live experiments use current public endpoints:

- Microsoft Planetary Computer STAC for **Sentinel-2 L2A** and **ESA WorldCover**.
- AWS Open Data for **Copernicus DEM GLO-30**.
- NOAA/AOML ERDDAP mirror for **NASA GPM IMERG V07**.

The notebook discovers assets dynamically where possible instead of relying on brittle hard-coded COG paths.

In [ ]:
SOURCE_REGISTRY = {
    "sentinel2": {
        "provider": "Microsoft Planetary Computer",
        "collection": "sentinel-2-l2a",
        "stac": "https://planetarycomputer.microsoft.com/api/stac/v1",
        "purpose": "surface-reflectance bands, cloud mask, mixed native resolutions"
    },
    "worldcover": {
        "provider": "Microsoft Planetary Computer / ESA",
        "collection": "esa-worldcover",
        "stac": "https://planetarycomputer.microsoft.com/api/stac/v1",
        "purpose": "nominal categorical raster and interpolation validity"
    },
    "copernicus_dem": {
        "provider": "AWS Open Data / Copernicus",
        "bucket": "https://copernicus-dem-30m.s3.amazonaws.com",
        "product": "Copernicus DEM GLO-30",
        "surface_model": "DSM",
        "horizontal_crs": "EPSG:4326",
        "vertical_datum": "EGM2008 / EPSG:3855",
        "purpose": "terrain semantics, DSM-vs-DTM, vertical datum"
    },
    "imerg": {
        "provider": "NOAA/AOML ERDDAP mirror of NASA GPM IMERG",
        "dataset": "precipitation_2024_v07",
        "base": "https://erddap.aoml.noaa.gov/hdb/erddap/griddap/precipitation_2024_v07",
        "units": "MM/Hr",
        "temporal_resolution": "30 minutes",
        "spatial_resolution": "0.1 degree",
        "purpose": "rate vs accumulation and temporal support"
    }
}

pd.DataFrame([
    {"source": k, **{kk: vv for kk, vv in v.items() if isinstance(vv, (str,int,float,bool))}}
    for k,v in SOURCE_REGISTRY.items()
])

# Part II — Scientific semantic model

The verifier receives only these three objects:

\[
(	ext{Artifacts}, 	ext{Operations}, 	ext{Scientific Intent})
\]

It never receives benchmark labels.

In [ ]:
@dataclass
class Artifact:
    name: str
    quantity: str
    unit: Optional[str] = None
    scale: str = "continuous"
    crs: Optional[str] = None
    resolution_m: Optional[float] = None
    temporal_resolution_h: Optional[float] = None
    processing_level: Optional[str] = None
    surface_model: Optional[str] = None
    vertical_datum: Optional[str] = None
    acquisition_time: Optional[str] = None
    available_time: Optional[str] = None
    provenance: Optional[str] = None
    band_role: Optional[str] = None
    nodata_semantics: Optional[str] = None
    uncertainty: Optional[float] = None
    allowed_values: Optional[List[float]] = None
    observed_min: Optional[float] = None
    observed_max: Optional[float] = None
    attrs: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Step:
    op: str
    params: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Intent:
    goal: str
    required_quantity: Optional[str] = None
    required_unit: Optional[str] = None
    decision_time: Optional[str] = None
    max_resolution_m: Optional[float] = None
    requires_bare_earth: bool = False
    requires_cloud_mask: bool = False
    max_uncertainty: Optional[float] = None
    temporal_tolerance_h: Optional[float] = None
    required_value_range: Optional[List[float]] = None

@dataclass
class BenchCase:
    case_id: str
    split: str
    tier: str
    family: str
    artifacts: List[Artifact]
    steps: List[Step]
    intent: Intent
    expected: str
    truth_reason: str

In [ ]:
AREA_UNITS = {"m2", "km2", "ha"}
LENGTH_UNITS = {"m", "km", "cm", "mm"}
RATE_UNITS = {"mm/h", "mm/hr", "MM/Hr"}
ACCUM_UNITS = {"mm", "cm", "m"}
DIMENSIONLESS = {"dimensionless", "1"}
CATEGORICAL_SCALES = {"categorical", "nominal", "ordinal", "binary"}
CONTINUOUS_INTERPOLATORS = {"bilinear", "cubic", "lanczos"}

UNIT_DIMENSION = {
    "m2": "area", "km2": "area", "ha": "area",
    "m": "length", "km": "length", "cm": "length", "mm": "length_or_accum",
    "mm/h": "rate", "mm/hr": "rate", "MM/Hr": "rate",
    "dimensionless": "dimensionless", "1": "dimensionless",
    "K": "temperature", "degC": "temperature",
}

def parse_dt(x):
    if not x:
        return None
    d = datetime.fromisoformat(str(x).replace("Z", "+00:00"))
    if d.tzinfo is None:
        d = d.replace(tzinfo=timezone.utc)
    return d

def unit_dimension(u):
    return UNIT_DIMENSION.get(u)

def artifact_copy(x):
    return replace(x, attrs=dict(x.attrs))

# Part III — Family-blind verifier

The verifier below contains generic scientific invariants.  
It does **not** branch on benchmark family names.

In [ ]:
def verify_science(artifacts: List[Artifact], steps: List[Step], intent: Intent):
    A = [artifact_copy(x) for x in artifacts]
    findings = []

    def add(code, verdict, message, step_index=None, evidence=None, repair=None):
        findings.append({
            "code": code,
            "verdict": verdict,
            "message": message,
            "step_index": step_index,
            "evidence": evidence or {},
            "repair": repair
        })

    # ---------- source evidence ----------
    for i, x in enumerate(A):
        if not x.provenance:
            add("PROVENANCE_MISSING", "UNPROVEN",
                f"Provenance is missing for artifact {i}: {x.name}.",
                evidence={"artifact": i},
                repair="Attach authoritative product/version/acquisition metadata.")

        if x.resolution_m is not None and x.resolution_m <= 0:
            add("RESOLUTION_INVALID", "REFUTED",
                "Spatial resolution must be positive.", evidence={"artifact": i})

        if x.observed_min is not None and x.observed_max is not None:
            if x.observed_min > x.observed_max:
                add("VALUE_RANGE_INVALID", "REFUTED",
                    "Observed minimum exceeds observed maximum.",
                    evidence={"artifact": i})

    # ---------- workflow semantics ----------
    for si, s in enumerate(steps):
        p = s.params
        op = s.op

        if op == "resample":
            i = p["artifact"]
            method = str(p.get("method", "nearest")).lower()
            target = float(p["target_resolution_m"])
            x = A[i]

            if x.scale in CATEGORICAL_SCALES and method in CONTINUOUS_INTERPOLATORS:
                add("CATEGORICAL_INTERPOLATION", "REFUTED",
                    f"{method} interpolation does not preserve {x.scale} identity.",
                    si,
                    {"measurement_scale": x.scale, "method": method},
                    "Use nearest-neighbour for identity preservation or mode/majority for aggregation.")

            if x.resolution_m is not None and target < x.resolution_m:
                gain = x.resolution_m / target
                if gain > 1.25 and not p.get("evidence_of_new_information", False):
                    add("UNSUPPORTED_RESOLUTION_GAIN", "REFUTED",
                        f"Grid spacing improves {gain:.2f}x without evidence of new information.",
                        si,
                        {"native_m": x.resolution_m, "target_m": target})
            x.resolution_m = target

        elif op == "temporal_aggregate":
            i = p["artifact"]
            x = A[i]
            method = str(p.get("method", "")).lower()
            duration_h = p.get("sample_duration_h")
            declared = p.get("declared_quantity")

            if x.unit in RATE_UNITS and declared == "accumulation":
                if method in {"mean", "average"}:
                    add("RATE_AVERAGED_AS_ACCUMULATION", "REFUTED",
                        "A rate is averaged but the result is declared as an accumulation.",
                        si,
                        {"unit": x.unit, "method": method})
                elif method == "sum" and duration_h not in {1, 1.0}:
                    add("RATE_SUM_WITHOUT_DURATION", "REFUTED",
                        "Sub-hourly rates are summed without multiplying by interval duration.",
                        si,
                        {"sample_duration_h": duration_h})
                elif method in {"integrate", "duration_weighted_sum"}:
                    x.quantity = "precipitation_accumulation"
                    x.unit = "mm"

        elif op == "area":
            i = p["artifact"]
            x = A[i]
            method = p.get("method", "planar")
            if x.crs in {"EPSG:4326", "OGC:CRS84", "CRS84"} and method == "planar":
                add("ANGULAR_COORDINATE_AREA", "REFUTED",
                    "Planar area is computed directly from angular coordinates.",
                    si,
                    {"crs": x.crs},
                    "Use geodesic area or an appropriate projected/equal-area CRS.")
            x.quantity = "area"
            x.unit = p.get("output_unit", "km2")

        elif op == "spectral_index":
            red = A[p["red_artifact"]]
            other = A[p["other_artifact"]]
            expected_other = p.get("expected_other_role")

            if red.band_role != "red":
                add("RED_ROLE_MISMATCH", "REFUTED",
                    f"Red-band role required; got {red.band_role}.", si)

            if expected_other and other.band_role != expected_other:
                add("SPECTRAL_ROLE_MISMATCH", "REFUTED",
                    f"{expected_other} role required; got {other.band_role}.", si)

            if (red.resolution_m is not None and other.resolution_m is not None
                and red.resolution_m != other.resolution_m
                and not p.get("explicit_alignment", False)):
                add("BAND_GRID_MISMATCH", "REFUTED",
                    "Spectral operands have different native resolutions without explicit alignment.",
                    si,
                    {"red_m": red.resolution_m, "other_m": other.resolution_m})

            if p.get("output_artifact") is not None:
                out = A[p["output_artifact"]]
                out.quantity = p.get("output_quantity", out.quantity)
                out.unit = "dimensionless"

        elif op == "require_surface_reflectance":
            x = A[p["artifact"]]
            if x.processing_level not in {"L2A", "surface_reflectance"}:
                add("PROCESSING_LEVEL_MISMATCH", "REFUTED",
                    f"Surface reflectance required; got {x.processing_level}.", si)

        elif op == "require_cloud_mask":
            x = A[p["artifact"]]
            if not bool(x.attrs.get("cloud_mask_applied", False)):
                add("CLOUD_MASK_MISSING", "REFUTED",
                    "Cloud-sensitive analysis has no recorded cloud mask.", si)

        elif op == "require_bare_earth":
            x = A[p["artifact"]]
            if x.surface_model == "DSM":
                add("DSM_USED_AS_DTM", "REFUTED",
                    "Bare-earth analysis is requested from a DSM.", si)
            elif x.surface_model is None:
                add("SURFACE_MODEL_UNKNOWN", "UNPROVEN",
                    "DSM/DTM semantics are not known.", si)

        elif op == "combine_vertical":
            x = A[p["a"]]
            y = A[p["b"]]
            if not x.vertical_datum or not y.vertical_datum:
                add("VERTICAL_DATUM_UNKNOWN", "UNPROVEN",
                    "Vertical-datum compatibility cannot be established.", si)
            elif x.vertical_datum != y.vertical_datum:
                add("VERTICAL_DATUM_MISMATCH", "REFUTED",
                    "Vertical quantities use incompatible datums.", si,
                    {"a": x.vertical_datum, "b": y.vertical_datum})

        elif op == "predict":
            decision = parse_dt(intent.decision_time)
            for i in p.get("features", range(len(A))):
                t = parse_dt(A[i].available_time or A[i].acquisition_time)
                if decision is not None and t is None:
                    add("FEATURE_TIME_UNKNOWN", "UNPROVEN",
                        "Feature availability time is unknown.", si, {"artifact": i})
                elif decision is not None and t > decision:
                    add("FUTURE_INFORMATION", "REFUTED",
                        "A model feature becomes available after the decision time.",
                        si,
                        {"artifact": i, "feature_time": str(t), "decision_time": str(decision)})

        elif op == "split":
            mode = p.get("mode")
            if mode == "spatial":
                sep = float(p.get("separation_m", 0))
                corr = float(p.get("dependence_range_m", 0))
                if corr > 0 and sep < corr:
                    add("SPATIAL_DEPENDENCE_LEAKAGE", "REFUTED",
                        "Train/test separation is below the estimated spatial dependence range.",
                        si,
                        {"separation_m": sep, "dependence_range_m": corr})
            elif mode == "temporal":
                train_end = parse_dt(p.get("train_end"))
                test_start = parse_dt(p.get("test_start"))
                if train_end and test_start and train_end >= test_start:
                    add("TEMPORAL_SPLIT_OVERLAP", "REFUTED",
                        "Training and testing periods overlap or are not chronologically ordered.", si)
            elif mode == "scene":
                overlap = float(p.get("scene_overlap_fraction", 0))
                if overlap > 0:
                    add("SCENE_OVERLAP_LEAKAGE", "REFUTED",
                        "The same scene content appears in both train and test partitions.",
                        si, {"overlap_fraction": overlap})

        elif op == "fill_nodata":
            x = A[p["artifact"]]
            fill = p.get("value")
            if (x.nodata_semantics in {"missing", "masked"}
                and fill == 0
                and p.get("zero_is_physically_valid", True)):
                add("NODATA_COLLAPSED_TO_ZERO", "REFUTED",
                    "Missing data are replaced by a physically meaningful zero.", si)

        elif op == "compare_sensors":
            x = A[p["a"]]
            y = A[p["b"]]
            tx = parse_dt(x.acquisition_time)
            ty = parse_dt(y.acquisition_time)
            tol = float(p.get("tolerance_h", intent.temporal_tolerance_h or 0))
            if not tx or not ty:
                add("ACQUISITION_TIME_UNKNOWN", "UNPROVEN",
                    "Multisensor acquisition-time compatibility is unknown.", si)
            else:
                gap = abs((tx - ty).total_seconds()) / 3600
                if gap > tol:
                    add("MULTISENSOR_TIME_MISMATCH", "REFUTED",
                        "Acquisition gap exceeds the declared scientific tolerance.",
                        si, {"gap_h": gap, "tolerance_h": tol})

        elif op == "convert_unit":
            x = A[p["artifact"]]
            src = x.unit
            dst = p["to"]
            ds, dd = unit_dimension(src), unit_dimension(dst)
            if ds is None or dd is None:
                add("UNIT_DIMENSION_UNKNOWN", "UNPROVEN",
                    "Unit dimensionality is not known.", si, {"from": src, "to": dst})
            elif ds != dd and not ({ds,dd} <= {"length","length_or_accum"}):
                add("UNIT_DIMENSION_MISMATCH", "REFUTED",
                    "Requested conversion changes physical dimension.", si,
                    {"from": src, "to": dst})
            else:
                x.unit = dst

        elif op == "assert_value_range":
            x = A[p["artifact"]]
            lo, hi = p["min"], p["max"]
            if x.observed_min is None or x.observed_max is None:
                add("VALUE_RANGE_UNOBSERVED", "UNPROVEN",
                    "Required physical bounds cannot be checked because observations are missing.", si)
            elif x.observed_min < lo or x.observed_max > hi:
                add("PHYSICAL_RANGE_VIOLATION", "REFUTED",
                    "Observed values violate declared physical bounds.",
                    si,
                    {"observed": [x.observed_min, x.observed_max], "required": [lo, hi]})

        elif op == "aggregate_categorical":
            x = A[p["artifact"]]
            method = p.get("method")
            if x.scale in CATEGORICAL_SCALES and method in {"mean", "median"}:
                add("CATEGORICAL_AGGREGATION_INVALID", "REFUTED",
                    f"{method} is not label-preserving for nominal classes.", si)

        elif op == "drop_quality_metadata":
            x = A[p["artifact"]]
            if p.get("required_downstream", False):
                add("QUALITY_METADATA_LOST", "REFUTED",
                    "A downstream-required quality flag is discarded.", si)

        elif op == "declare_uncertainty":
            A[p["artifact"]].uncertainty = p.get("value")

    # ---------- purpose / postconditions ----------
    if intent.required_unit:
        result_like = [x for x in A if x.quantity == intent.required_quantity] if intent.required_quantity else A
        if result_like:
            known_units = {x.unit for x in result_like if x.unit}
            if known_units and intent.required_unit not in known_units:
                req_dim = unit_dimension(intent.required_unit)
                actual_dims = {unit_dimension(u) for u in known_units}
                if req_dim not in actual_dims:
                    add("OUTPUT_DIMENSION_MISMATCH", "REFUTED",
                        "Output physical dimension is incompatible with the scientific intent.",
                        evidence={"required": intent.required_unit, "actual": sorted(known_units)})

    if intent.max_resolution_m is not None:
        for i,x in enumerate(A):
            if x.resolution_m is None:
                add("RESOLUTION_UNKNOWN", "UNPROVEN",
                    "Required spatial support cannot be checked.", evidence={"artifact": i})
            elif x.resolution_m > intent.max_resolution_m:
                add("RESOLUTION_TOO_COARSE", "REFUTED",
                    "Available spatial resolution is coarser than the declared requirement.",
                    evidence={"artifact": i, "resolution_m": x.resolution_m,
                              "required_m": intent.max_resolution_m})

    if intent.max_uncertainty is not None:
        known = [x.uncertainty for x in A if x.uncertainty is not None]
        if not known:
            add("UNCERTAINTY_UNKNOWN", "UNPROVEN",
                "The intent defines an uncertainty limit but no uncertainty evidence is available.")
        elif max(known) > intent.max_uncertainty:
            add("UNCERTAINTY_EXCEEDS_LIMIT", "REFUTED",
                "Output uncertainty exceeds the declared fitness-for-purpose limit.",
                evidence={"max_observed": max(known), "limit": intent.max_uncertainty})

    if intent.required_value_range:
        lo,hi = intent.required_value_range
        relevant = [x for x in A if x.quantity == intent.required_quantity] if intent.required_quantity else A
        for x in relevant:
            if x.observed_min is None or x.observed_max is None:
                add("INTENT_RANGE_UNPROVEN", "UNPROVEN",
                    "Intent-specific physical bounds cannot be checked.")
            elif x.observed_min < lo or x.observed_max > hi:
                add("INTENT_RANGE_REFUTED", "REFUTED",
                    "Output violates the physical range required by the declared scientific intent.")

    if any(f["verdict"] == "REFUTED" for f in findings):
        verdict = "REFUTED"
    elif any(f["verdict"] == "UNPROVEN" for f in findings):
        verdict = "UNPROVEN"
    else:
        verdict = "PROVEN"

    return {"verdict": verdict, "findings": findings, "final_artifacts": [asdict(x) for x in A]}

### Independence check

The benchmark wrapper calls the verifier with **only artifacts, steps, and intent**.

In [ ]:
def evaluate_case(case: BenchCase):
    return verify_science(case.artifacts, case.steps, case.intent)

print("Verifier inputs:", verify_science.__annotations__.keys())
print("Benchmark family is not an input to verify_science.")

# Part IV — Real Sentinel-2 experiment

This cell fixes the v2 `0 valid pixels` problem.

Workflow:

1. Search the Planetary Computer for low-cloud Sentinel-2 L2A scenes over a fixed research ROI.
2. Sort by cloud cover.
3. Scan candidate COG windows.
4. Pick a window only when both B04 and B08 contain enough valid non-zero pixels.
5. Read SCL over the same bounds and apply a nearest-neighbour cloud/quality mask.
6. Compute real NDVI.
7. Also compute a deliberately wrong SWIR-based pseudo-index for comparison.

In [ ]:
def install_live_packages():
    if not LIVE_DATA:
        print("LIVE_DATA=False — live packages not installed.")
        return
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "pystac-client", "planetary-computer", "rasterio", "scipy",
        "requests", "matplotlib"
    ])
    print("Live packages installed.")

install_live_packages()

In [ ]:
# Fixed scientific demonstration ROI: Bengaluru metropolitan region, India.
# This is only an example study region; change it if you want a different site.
S2_ROI = [77.45, 12.85, 77.75, 13.15]
S2_DATE_RANGE = "2024-01-01/2025-12-31"

def get_s2_candidates(max_cloud=20, max_items=20):
    import pystac_client, planetary_computer
    cat = pystac_client.Client.open(
        SOURCE_REGISTRY["sentinel2"]["stac"],
        modifier=planetary_computer.sign_inplace
    )
    search = cat.search(
        collections=[SOURCE_REGISTRY["sentinel2"]["collection"]],
        bbox=S2_ROI,
        datetime=S2_DATE_RANGE,
        query={"eo:cloud_cover": {"lt": max_cloud}},
        max_items=max_items
    )
    items = list(search.items())
    items.sort(key=lambda x: float(x.properties.get("eo:cloud_cover", 999)))
    return items

def find_usable_s2_window(item, size=512, min_valid_fraction=0.70):
    import rasterio, planetary_computer
    from rasterio.windows import Window

    u4 = planetary_computer.sign(item.assets["B04"].href)
    u8 = planetary_computer.sign(item.assets["B08"].href)

    env = {
        "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF",
        "GDAL_HTTP_MULTIRANGE": "YES",
        "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES"
    }

    with rasterio.Env(**env), rasterio.open(u4) as r4, rasterio.open(u8) as r8:
        row_positions = np.linspace(0, max(0, r4.height-size), 8, dtype=int)
        col_positions = np.linspace(0, max(0, r4.width-size), 8, dtype=int)

        candidates = [(r,c) for r in row_positions for c in col_positions]
        random.Random(SEED).shuffle(candidates)

        best = None
        for row, col in candidates:
            w = Window(col, row, min(size, r4.width-col), min(size, r4.height-row))
            b4 = r4.read(1, window=w, masked=True)
            b8 = r8.read(1, window=w, masked=True)

            valid = (~np.ma.getmaskarray(b4)) & (~np.ma.getmaskarray(b8)) & (b4 > 0) & (b8 > 0)
            frac = float(valid.mean())

            if best is None or frac > best["valid_fraction"]:
                best = {"window": w, "valid_fraction": frac}

            if frac >= min_valid_fraction:
                return best

        return best

def run_sentinel2_real_experiment():
    if not LIVE_DATA:
        red = np.array([[1000,1200,1500],[900,1800,2200],[500,700,1000]], dtype=np.float32)
        nir = np.array([[2500,2600,2400],[1800,2200,1900],[1200,900,2000]], dtype=np.float32)
        ndvi = (nir-red)/(nir+red)
        return {
            "mode": "offline_fixture",
            "valid_fraction": 1.0,
            "ndvi_min": float(ndvi.min()),
            "ndvi_median": float(np.median(ndvi)),
            "ndvi_max": float(ndvi.max()),
            "note": "Enable LIVE_DATA for real Sentinel-2 pixels."
        }

    import rasterio, planetary_computer
    from rasterio.windows import bounds
    from rasterio.transform import from_bounds
    from rasterio.warp import reproject, Resampling

    items = get_s2_candidates()
    if not items:
        raise RuntimeError("No Sentinel-2 candidates found for the ROI/date/cloud criteria.")

    chosen = None
    for item in items:
        window_info = find_usable_s2_window(item)
        if window_info and window_info["valid_fraction"] >= 0.50:
            chosen = (item, window_info)
            if window_info["valid_fraction"] >= 0.70:
                break

    if chosen is None:
        raise RuntimeError("Could not find a Sentinel-2 window containing usable pixels.")

    item, wi = chosen
    w = wi["window"]

    u4 = planetary_computer.sign(item.assets["B04"].href)
    u8 = planetary_computer.sign(item.assets["B08"].href)
    u11 = planetary_computer.sign(item.assets["B11"].href)
    uscl = planetary_computer.sign(item.assets["SCL"].href)

    env = {
        "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF",
        "GDAL_HTTP_MULTIRANGE": "YES",
        "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES"
    }

    with rasterio.Env(**env), rasterio.open(u4) as r4, rasterio.open(u8) as r8:
        red = r4.read(1, window=w).astype(np.float32)
        nir = r8.read(1, window=w).astype(np.float32)
        dst_transform = r4.window_transform(w)
        dst_crs = r4.crs
        h,wid = red.shape
        bnds = rasterio.windows.bounds(w, r4.transform)

    # Reproject 20 m SCL to the selected 10 m window.
    scl10 = np.zeros((h,wid), dtype=np.uint8)
    with rasterio.Env(**env), rasterio.open(uscl) as rscl:
        reproject(
            source=rasterio.band(rscl, 1),
            destination=scl10,
            src_transform=rscl.transform,
            src_crs=rscl.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

    # Reproject 20 m B11 onto the same 10 m analysis grid for the *wrong-band* experiment.
    swir10 = np.zeros((h,wid), dtype=np.float32)
    with rasterio.Env(**env), rasterio.open(u11) as r11:
        reproject(
            source=rasterio.band(r11, 1),
            destination=swir10,
            src_transform=r11.transform,
            src_crs=r11.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

    bad_scl = np.isin(scl10, [0,1,3,8,9,10,11])
    valid = (red > 0) & (nir > 0) & (~bad_scl)

    ndvi = np.full(red.shape, np.nan, dtype=np.float32)
    denom = nir + red
    np.divide(nir-red, denom, out=ndvi, where=valid & (denom != 0))

    wrong = np.full(red.shape, np.nan, dtype=np.float32)
    wrong_denom = swir10 + red
    np.divide(swir10-red, wrong_denom, out=wrong, where=valid & (wrong_denom != 0))

    return {
        "mode": "live_real_sentinel2",
        "item_id": item.id,
        "cloud_cover_percent": float(item.properties.get("eo:cloud_cover", np.nan)),
        "window_valid_fraction_before_scl": wi["valid_fraction"],
        "valid_fraction_after_scl": float(np.isfinite(ndvi).mean()),
        "ndvi_min": float(np.nanmin(ndvi)),
        "ndvi_median": float(np.nanmedian(ndvi)),
        "ndvi_max": float(np.nanmax(ndvi)),
        "wrong_index_median": float(np.nanmedian(wrong)),
        "mean_absolute_ndvi_vs_wrong_difference": float(np.nanmean(np.abs(ndvi-wrong))),
        "analysis_bounds": tuple(float(x) for x in bnds),
        "crs": str(dst_crs)
    }

sentinel_result = run_sentinel2_real_experiment()
sentinel_result

# Part V — Real WorldCover interpolation experiment

The v2 experiment was invalid because Rasterio detected the paletted categorical band and silently fell back to nearest-neighbour.

v3 avoids that completely:

1. Read the real WorldCover classes normally.
2. Convert the returned class matrix to `float32`.
3. Perform nearest-neighbour interpolation with `order=0`.
4. Perform true continuous bilinear interpolation with `order=1`.
5. Measure how many output pixels are not valid WorldCover class codes.

In [ ]:
WORLDCOVER_CLASSES = np.array([10,20,30,40,50,60,70,80,90,95,100], dtype=np.float32)

def get_worldcover_item():
    import pystac_client, planetary_computer
    cat = pystac_client.Client.open(
        SOURCE_REGISTRY["worldcover"]["stac"],
        modifier=planetary_computer.sign_inplace
    )
    search = cat.search(
        collections=[SOURCE_REGISTRY["worldcover"]["collection"]],
        bbox=S2_ROI,
        max_items=10
    )
    items = list(search.items())
    if not items:
        raise RuntimeError("No ESA WorldCover item found for the ROI.")
    return items[0]

def choose_worldcover_asset(item):
    # Planetary Computer WorldCover items normally use `map`.
    for key in ["map", "data", "classification"]:
        if key in item.assets:
            return key
    # Fall back to first TIFF-like asset.
    for key, asset in item.assets.items():
        href = asset.href.lower()
        if ".tif" in href:
            return key
    raise RuntimeError(f"No raster asset found. Assets: {list(item.assets)}")

def run_worldcover_real_experiment():
    if not LIVE_DATA:
        src = np.array([
            [10,10,10,40,40,40],
            [10,10,40,40,40,40],
            [10,80,80,40,40,50],
            [80,80,80,80,50,50],
            [80,80,90,90,50,50],
            [80,90,90,90,50,50]
        ], dtype=np.float32)
        from scipy.ndimage import zoom
        nn = zoom(src, 0.63, order=0, prefilter=False)
        bl = zoom(src, 0.63, order=1, prefilter=False)
        invalid = ~np.isin(bl, WORLDCOVER_CLASSES)
        return {
            "mode": "offline_fixture",
            "source_classes": sorted(np.unique(src).tolist()),
            "nearest_classes": sorted(np.unique(nn).tolist()),
            "bilinear_unique_count": int(np.unique(bl).size),
            "invalid_bilinear_fraction": float(invalid.mean()),
            "invalid_bilinear_examples": sorted(np.unique(bl[invalid]).round(3).tolist())[:20]
        }

    import rasterio, planetary_computer
    from scipy.ndimage import zoom
    from rasterio.windows import Window

    item = get_worldcover_item()
    asset_key = choose_worldcover_asset(item)
    href = planetary_computer.sign(item.assets[asset_key].href)

    env = {
        "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF",
        "GDAL_HTTP_MULTIRANGE": "YES",
        "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES"
    }

    with rasterio.Env(**env), rasterio.open(href) as src:
        size = min(768, src.height, src.width)
        rows = np.linspace(0, max(0,src.height-size), 6, dtype=int)
        cols = np.linspace(0, max(0,src.width-size), 6, dtype=int)
        best = None

        for r in rows:
            for c in cols:
                arr = src.read(1, window=Window(c,r,size,size))
                valid = np.isin(arr, WORLDCOVER_CLASSES)
                diversity = len(np.unique(arr[valid])) if valid.any() else 0
                score = float(valid.mean()) + 0.03*diversity
                if best is None or score > best[0]:
                    best = (score, arr, r, c, valid.mean(), diversity)

        _, arr, r, c, valid_frac, diversity = best

    arrf = arr.astype(np.float32)

    # Explicit numerical interpolation; Rasterio cannot replace our method.
    scale = 0.47
    nearest = zoom(arrf, scale, order=0, prefilter=False)
    bilinear = zoom(arrf, scale, order=1, prefilter=False)

    nearest_valid = np.isin(nearest, WORLDCOVER_CLASSES)

    # Continuous interpolation outputs are floats; exact class codes are the only valid labels.
    bilinear_is_class = np.isin(bilinear, WORLDCOVER_CLASSES)
    invalid = ~bilinear_is_class

    return {
        "mode": "live_real_worldcover",
        "item_id": item.id,
        "asset_key": asset_key,
        "source_valid_fraction": float(valid_frac),
        "source_class_diversity": int(diversity),
        "source_classes": sorted(np.unique(arr[np.isin(arr,WORLDCOVER_CLASSES)]).tolist()),
        "nearest_valid_fraction": float(nearest_valid.mean()),
        "bilinear_invalid_fraction": float(invalid.mean()),
        "bilinear_invalid_examples": sorted(np.unique(bilinear[invalid]).round(3).tolist())[:25],
        "source_shape": tuple(arr.shape),
        "output_shape": tuple(bilinear.shape)
    }

worldcover_result = run_worldcover_real_experiment()
worldcover_result

# Part VI — IMERG and Copernicus DEM real-data checks

These retain the working v2 approach but harden column/tile discovery.

In [ ]:
def imerg_query_url(
    date="2024-07-01",
    lat_min=12.90, lat_max=13.10,
    lon_min=77.50, lon_max=77.70
):
    base = SOURCE_REGISTRY["imerg"]["base"]
    start = f"{date}T00:00:00Z"
    stop  = f"{date}T23:30:00Z"
    q = (
        f"precipitation[({start}):1:({stop})]"
        f"[({lat_min}):1:({lat_max})]"
        f"[({lon_min}):1:({lon_max})]"
    )
    return base + ".csv?" + q

def detect_column(df, contains):
    cs = [c for c in df.columns if contains.lower() in c.lower()]
    if not cs:
        raise KeyError(f"Could not detect column containing {contains!r}. Columns={list(df.columns)}")
    return cs[0]

def run_imerg_real_experiment():
    if not LIVE_DATA:
        rates = np.array([0]*10 + [0.5,1,2,4,8,12,10,6,4,2,1] + [0]*18 + [1,2,3,4,2,1] + [0]*3, dtype=float)[:48]
        return {
            "mode": "offline_fixture",
            "correct_accumulation_mm": float(np.sum(rates*0.5)),
            "naive_rate_mean_mm_per_h": float(np.mean(rates)),
            "raw_rate_sum": float(np.sum(rates))
        }

    df = pd.read_csv(imerg_query_url())
    pcol = detect_column(df, "precip")
    latcol = detect_column(df, "lat")
    loncol = detect_column(df, "lon")

    df[pcol] = pd.to_numeric(df[pcol], errors="coerce")
    clean = df.dropna(subset=[pcol,latcol,loncol]).copy()
    clean["interval_accum_mm"] = clean[pcol] * 0.5

    by_cell = clean.groupby([latcol,loncol])["interval_accum_mm"].sum()

    return {
        "mode": "live_real_imerg",
        "rows": int(len(clean)),
        "grid_cells": int(len(by_cell)),
        "mean_daily_accumulation_mm": float(by_cell.mean()),
        "min_daily_accumulation_mm": float(by_cell.min()),
        "max_daily_accumulation_mm": float(by_cell.max()),
        "naive_rate_mean_mm_per_h": float(clean[pcol].mean()),
        "precipitation_column": pcol
    }

imerg_result = run_imerg_real_experiment()
imerg_result

In [ ]:
def copdem_prefix(lat_deg, lon_deg):
    ns = "N" if lat_deg >= 0 else "S"
    ew = "E" if lon_deg >= 0 else "W"
    return f"Copernicus_DSM_COG_10_{ns}{abs(int(lat_deg)):02d}_00_{ew}{abs(int(lon_deg)):03d}_00_DEM"

def run_copdem_real_experiment(lat=12, lon=77):
    if not LIVE_DATA:
        return {
            "mode":"offline_specification",
            "surface_model":"DSM",
            "horizontal_crs":"EPSG:4326",
            "vertical_datum":"EGM2008 / EPSG:3855",
            "resolution_m_approx":30
        }

    import requests, rasterio
    from rasterio.windows import Window

    base = SOURCE_REGISTRY["copernicus_dem"]["bucket"]
    tiles = set(requests.get(base + "/tileList.txt", timeout=60).text.splitlines())
    requested = copdem_prefix(lat,lon)

    if requested not in tiles:
        # deterministic land fallback rather than opening a nonexistent URL
        candidates = sorted([x for x in tiles if x.startswith("Copernicus_DSM_COG_10_N")])
        if not candidates:
            raise RuntimeError("No public GLO-30 tiles were returned by tileList.txt.")
        requested = candidates[0]

    url = f"{base}/{requested}/{requested}.tif"

    env = {
        "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF",
        "GDAL_HTTP_MULTIRANGE": "YES"
    }

    with rasterio.Env(**env), rasterio.open(url) as src:
        size = min(256,src.height,src.width)
        w = Window(max(0,(src.width-size)//2), max(0,(src.height-size)//2), size, size)
        z = src.read(1,window=w,masked=True).astype(float)
        vals = z.compressed()

        return {
            "mode":"live_real_copdem",
            "tile":requested,
            "crs":str(src.crs),
            "resolution":tuple(float(v) for v in src.res),
            "valid_pixels":int(vals.size),
            "min_elevation_m":float(np.min(vals)),
            "median_elevation_m":float(np.median(vals)),
            "max_elevation_m":float(np.max(vals)),
            "surface_model":"DSM",
            "vertical_datum":"EGM2008 / EPSG:3855"
        }

copdem_result = run_copdem_real_experiment()
copdem_result

# Part VII — SilentGeoBench v3 generators

### Split policy

- **development**: individual error families used while building the verifier.
- **heldout-family**: families withheld from development.
- **blind-composition**: multi-step workflows generated separately after the single-family generators.

### Uniqueness policy

Cases are accepted only when their scientific configuration hash is new.

In [ ]:
DEV_FAMILIES = [
    "categorical_interpolation",
    "rate_mean_as_accumulation",
    "angular_area",
    "spectral_role",
    "band_grid_mismatch",
    "cloud_mask_missing",
    "processing_level",
    "DSM_as_DTM",
    "future_information",
    "spatial_split_leakage",
    "nodata_zero",
    "unsupported_resolution_gain",
]

HELDOUT_FAMILIES = [
    "rate_sum_without_duration",
    "vertical_datum_mismatch",
    "vertical_datum_unknown",
    "multisensor_time_mismatch",
    "missing_provenance",
    "uncertainty_missing",
    "unit_dimension_mismatch",
    "physical_range_violation",
    "categorical_mean_aggregation",
    "temporal_split_overlap",
    "scene_overlap_leakage",
    "quality_metadata_loss",
]

ALL_SINGLE_FAMILIES = DEV_FAMILIES + HELDOUT_FAMILIES

def scientific_signature(artifacts, steps, intent):
    payload = {
        "artifacts": [asdict(x) for x in artifacts],
        "steps": [asdict(x) for x in steps],
        "intent": asdict(intent)
    }
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()

def A(**kw):
    base = dict(
        name="artifact",
        quantity="unknown",
        unit=None,
        scale="continuous",
        crs="EPSG:32643",
        resolution_m=10.0,
        provenance="authoritative-product"
    )
    base.update(kw)
    return Artifact(**base)

In [ ]:
def sample_single_case(family, rng, split):
    valid = rng.random() < 0.30
    tier = rng.choice(["atomic","compositional","adversarial"])
    truth = ""

    if family == "categorical_interpolation":
        native = rng.choice([10,20,30])

        if valid:
            # Valid categorical resampling must preserve labels AND must not
            # claim unsupported finer physical resolution.
            target_choices = [x for x in [10,15,20,25,30,35,45,60,90] if x >= native]
            target = rng.choice(target_choices)
            method = rng.choice(["nearest","mode"])
        else:
            target = rng.choice([15,25,35,45,60,90])
            method = rng.choice(["bilinear","cubic","lanczos"])

        arts=[A(name=f"landcover-{native}",quantity="land_cover",scale="categorical",
                crs="EPSG:4326",resolution_m=native,allowed_values=[10,20,30,40,50,60,70,80,90,95,100])]

        steps=[Step("resample",{"artifact":0,"method":method,"target_resolution_m":target,
                                "evidence_of_new_information": target >= native})]

        truth = (
            "label-preserving categorical resampling without unsupported resolution gain"
            if valid else
            "continuous interpolation creates non-label values"
        )

    elif family == "rate_mean_as_accumulation":
        duration = rng.choice([0.25,0.5])
        method = "integrate" if valid else rng.choice(["mean","average"])
        arts=[A(name="IMERG",quantity="precipitation_rate",unit="mm/h",crs="EPSG:4326",
                resolution_m=10000,temporal_resolution_h=duration)]
        steps=[Step("temporal_aggregate",{"artifact":0,"method":method,"sample_duration_h":duration,
                                          "declared_quantity":"accumulation"})]
        truth = "rate integrated over time" if valid else "rate averaged but declared as accumulation"

    elif family == "rate_sum_without_duration":
        duration=rng.choice([0.1,0.25,0.5])
        method="integrate" if valid else "sum"
        arts=[A(name="precip-rate",quantity="precipitation_rate",unit="mm/h",
                crs="EPSG:4326",temporal_resolution_h=duration,resolution_m=10000)]
        steps=[Step("temporal_aggregate",{"artifact":0,"method":method,"sample_duration_h":duration,
                                          "declared_quantity":"accumulation"})]
        truth="duration-aware integration" if valid else "sub-hourly rates summed without duration"

    elif family == "angular_area":
        crs = rng.choice(["EPSG:32643","EPSG:32644","EPSG:6933"]) if valid else "EPSG:4326"
        arts=[A(name="flood-polygon",quantity="flood_extent",scale="binary",unit=None,crs=crs,
                resolution_m=rng.choice([10,20,30]))]
        steps=[Step("area",{"artifact":0,"method":"planar","output_unit":"km2"})]
        truth="projected area computation" if valid else "square degrees mislabeled as metric area"

    elif family in {"spectral_role","band_grid_mismatch"}:
        other_role="nir" if valid or family=="band_grid_mismatch" else rng.choice(["swir16","green","rededge"])
        other_res=10 if valid or family=="spectral_role" else rng.choice([20,60])
        align=True if valid or family=="spectral_role" else False
        arts=[
            A(name="red",quantity="reflectance",processing_level="L2A",band_role="red",resolution_m=10),
            A(name="other",quantity="reflectance",processing_level="L2A",band_role=other_role,resolution_m=other_res)
        ]
        steps=[Step("spectral_index",{"red_artifact":0,"other_artifact":1,
                                      "expected_other_role":"nir","explicit_alignment":align})]
        truth="correct spectral operands" if valid else ("wrong spectral role" if family=="spectral_role" else "mixed grids without alignment")

    elif family == "cloud_mask_missing":
        cloud_applied = valid
        arts=[A(name="S2-L2A",quantity="surface_reflectance",processing_level="L2A",
                band_role="red",attrs={"cloud_mask_applied":cloud_applied})]
        steps=[Step("require_cloud_mask",{"artifact":0})]
        truth="cloud mask recorded" if valid else "cloud-sensitive analysis lacks cloud mask"

    elif family == "processing_level":
        level="L2A" if valid else rng.choice(["L1C","TOA","raw_DN"])
        arts=[A(name="optical",quantity="reflectance",processing_level=level,band_role="red")]
        steps=[Step("require_surface_reflectance",{"artifact":0})]
        truth="surface reflectance supplied" if valid else "wrong processing level"

    elif family == "DSM_as_DTM":
        model="DTM" if valid else "DSM"
        arts=[A(name="terrain",quantity="elevation",unit="m",resolution_m=30,
                surface_model=model,vertical_datum="EGM2008")]
        steps=[Step("require_bare_earth",{"artifact":0})]
        truth="bare-earth DTM" if valid else "surface model used as bare earth"

    elif family in {"vertical_datum_mismatch","vertical_datum_unknown"}:
        d1="EGM2008"
        if valid: d2="EGM2008"
        elif family=="vertical_datum_mismatch": d2=rng.choice(["EGM96","WGS84_ellipsoid"])
        else: d2=None
        arts=[
            A(name="terrain",quantity="elevation",unit="m",vertical_datum=d1),
            A(name="water",quantity="water_level",unit="m",vertical_datum=d2)
        ]
        steps=[Step("combine_vertical",{"a":0,"b":1})]
        truth="compatible vertical datums" if valid else ("incompatible vertical datums" if d2 else "vertical datum missing")

    elif family == "future_information":
        decision=datetime(2024,8,1,12,tzinfo=timezone.utc)
        delta=rng.choice([1,3,6,12,24])
        ft=decision-timedelta(hours=delta) if valid else decision+timedelta(hours=delta)
        arts=[A(name="rain-feature",quantity="rainfall",unit="mm",available_time=ft.isoformat())]
        steps=[Step("predict",{"features":[0]})]
        intent=Intent("flood forecast",decision_time=decision.isoformat())
        return arts,steps,intent,("PROVEN" if valid else "REFUTED"),tier,("pre-decision feature" if valid else "future feature leakage")

    elif family == "spatial_split_leakage":
        corr=rng.randint(400,6000)
        sep=rng.randint(corr, corr+5000) if valid else rng.randint(0,max(0,corr-1))
        arts=[A(name="samples",quantity="training_samples")]
        steps=[Step("split",{"mode":"spatial","separation_m":sep,"dependence_range_m":corr})]
        truth="spatial separation exceeds dependence range" if valid else "spatially dependent train/test samples"

    elif family == "temporal_split_overlap":
        test_start=datetime(2024,7,1,tzinfo=timezone.utc)
        train_end=test_start-timedelta(days=rng.randint(1,30)) if valid else test_start+timedelta(days=rng.randint(0,10))
        arts=[A(name="records",quantity="time_series")]
        steps=[Step("split",{"mode":"temporal","train_end":train_end.isoformat(),"test_start":test_start.isoformat()})]
        truth="strict chronological holdout" if valid else "training overlaps test period"

    elif family == "scene_overlap_leakage":
        overlap=0.0 if valid else round(rng.uniform(0.01,0.8),3)
        arts=[A(name="chips",quantity="image_samples")]
        steps=[Step("split",{"mode":"scene","scene_overlap_fraction":overlap})]
        truth="scene-disjoint split" if valid else "same scene content crosses partitions"

    elif family == "nodata_zero":
        fill=-9999 if valid else 0
        arts=[A(name="rain",quantity="rainfall",unit="mm",nodata_semantics="missing")]
        steps=[Step("fill_nodata",{"artifact":0,"value":fill,"zero_is_physically_valid":True})]
        truth="missingness preserved" if valid else "missing values collapsed to valid physical zero"

    elif family == "unsupported_resolution_gain":
        native=rng.choice([20,30,50,100])
        if valid:
            target=rng.choice([native,native*2,native*3])
            evidence=False
        else:
            target=native/rng.choice([2,3,4])
            evidence=False
        arts=[A(name="raster",quantity="surface_property",resolution_m=native)]
        steps=[Step("resample",{"artifact":0,"method":"bilinear","target_resolution_m":target,
                                "evidence_of_new_information":evidence})]
        truth="no unsupported information gain" if valid else "interpolation presented as new physical detail"

    elif family == "multisensor_time_mismatch":
        t0=datetime(2024,7,1,10,tzinfo=timezone.utc)
        tol=rng.choice([3,6,12,24])
        gap=rng.randint(0,tol) if valid else rng.randint(tol+1,tol+72)
        arts=[
            A(name="S1",quantity="backscatter",acquisition_time=t0.isoformat()),
            A(name="S2",quantity="reflectance",acquisition_time=(t0+timedelta(hours=gap)).isoformat())
        ]
        steps=[Step("compare_sensors",{"a":0,"b":1,"tolerance_h":tol})]
        truth="sensor dates within declared tolerance" if valid else "sensor dates exceed declared tolerance"

    elif family == "missing_provenance":
        prov="official-product" if valid else None
        arts=[A(name="unknown-input",quantity="elevation",provenance=prov)]
        steps=[]
        truth="source provenance present" if valid else "source provenance missing"

    elif family == "uncertainty_missing":
        unc=round(rng.uniform(0.005,0.04),4) if valid else None
        arts=[A(name="hazard",quantity="hazard_probability",unit="dimensionless",uncertainty=unc)]
        steps=[]
        intent=Intent("operational decision",max_uncertainty=0.05)
        return arts,steps,intent,("PROVEN" if valid else "UNPROVEN"),tier,("uncertainty within tolerance" if valid else "uncertainty evidence missing")

    elif family == "unit_dimension_mismatch":
        src="m"
        dst=rng.choice(["km","cm"]) if valid else rng.choice(["km2","K"])
        arts=[A(name="distance",quantity="distance",unit=src)]
        steps=[Step("convert_unit",{"artifact":0,"to":dst})]
        truth="dimension-preserving conversion" if valid else "conversion crosses physical dimensions"

    elif family == "physical_range_violation":
        if valid:
            lo,hi=sorted([rng.uniform(-1,0),rng.uniform(0,1)])
            lo=max(-1,lo); hi=min(1,hi)
        else:
            lo,hi=rng.choice([(-1.4,0.6),(-0.8,1.3),(-2,2)])
        arts=[A(name="index",quantity="NDVI",unit="dimensionless",
                observed_min=lo,observed_max=hi)]
        steps=[Step("assert_value_range",{"artifact":0,"min":-1,"max":1})]
        truth="values stay in physical NDVI range" if valid else "values violate physical NDVI range"

    elif family == "categorical_mean_aggregation":
        method="mode" if valid else rng.choice(["mean","median"])
        arts=[A(name="landcover",quantity="land_cover",scale="categorical",
                allowed_values=[10,20,30,40,50,60,70,80,90,95,100])]
        steps=[Step("aggregate_categorical",{"artifact":0,"method":method})]
        truth="categorical aggregation is label-preserving" if valid else "nominal labels aggregated numerically"

    elif family == "quality_metadata_loss":
        required=not valid
        arts=[A(name="reflectance",quantity="surface_reflectance",
                attrs={"quality_layer":"SCL"})]
        steps=[Step("drop_quality_metadata",{"artifact":0,"required_downstream":required})]
        truth="quality metadata not needed downstream" if valid else "required quality evidence discarded"

    else:
        raise ValueError(family)

    intent = Intent(family)
    expected = "PROVEN" if valid else ("UNPROVEN" if family in {"vertical_datum_unknown","missing_provenance"} else "REFUTED")
    return arts,steps,intent,expected,tier,truth

## Generate unique single-family cases

The loop rejects duplicate scientific signatures until each family reaches its requested count.

In [ ]:

FAMILY_TARGETS = {
    # development
    "categorical_interpolation": 16,
    "rate_mean_as_accumulation": 6,
    "angular_area": 12,
    "spectral_role": 4,
    "band_grid_mismatch": 3,
    "cloud_mask_missing": 2,
    "processing_level": 4,
    "DSM_as_DTM": 2,
    "future_information": 10,
    "spatial_split_leakage": 16,
    "nodata_zero": 2,
    "unsupported_resolution_gain": 16,

    # held-out families
    "rate_sum_without_duration": 6,
    "vertical_datum_mismatch": 3,
    "vertical_datum_unknown": 2,
    "multisensor_time_mismatch": 16,
    "missing_provenance": 2,
    "uncertainty_missing": 16,
    "unit_dimension_mismatch": 4,
    "physical_range_violation": 16,
    "categorical_mean_aggregation": 3,
    "temporal_split_overlap": 16,
    "scene_overlap_leakage": 16,
    "quality_metadata_loss": 2,
}

def unique_cases_for_family(family, n, split, seed_offset):
    rng = random.Random(SEED + seed_offset)
    out=[]
    seen=set()
    attempts=0

    while len(out)<n and attempts<50000:
        attempts+=1
        arts,steps,intent,expected,tier,truth = sample_single_case(family,rng,split)
        sig=scientific_signature(arts,steps,intent)
        if sig in seen:
            continue
        seen.add(sig)
        out.append(BenchCase(
            case_id=f"SGX3_{split.upper()}_{family}_{len(out):03d}",
            split=split,tier=tier,family=family,
            artifacts=arts,steps=steps,intent=intent,
            expected=expected,truth_reason=truth
        ))

    if len(out) != n:
        raise RuntimeError(
            f"{family}: could only generate {len(out)} unique cases, expected {n}."
        )
    return out

single_cases=[]
for j,fam in enumerate(DEV_FAMILIES):
    single_cases += unique_cases_for_family(
        fam, FAMILY_TARGETS[fam], "development", j*1000
    )

for j,fam in enumerate(HELDOUT_FAMILIES):
    single_cases += unique_cases_for_family(
        fam, FAMILY_TARGETS[fam], "heldout-family", 50000+j*1000
    )

print("single-family cases:",len(single_cases))
print("unique single-family signatures:",
      len({scientific_signature(c.artifacts,c.steps,c.intent) for c in single_cases}))
print(pd.Series([c.split for c in single_cases]).value_counts())


# Part VIII — Blind composition cases

These are produced by a separate generator that combines operations.  
Their labels are assigned by construction flags, not by calling SciGeoGuard-X.

In [ ]:

BLIND_TEMPLATES = [
    "flood_remote_sensing",
    "landslide_forecast",
    "landcover_change",
    "flood_depth",
    "eo_ml_validation"
]

def make_blind_case(template, i, rng):
    violations=[]
    unresolved=[]
    arts=[]
    steps=[]

    if template=="flood_remote_sensing":
        crs = rng.choice(["EPSG:32643","EPSG:6933","EPSG:4326"])
        area_method = rng.choice(["planar","geodesic"])
        cloud = rng.choice([True,False])
        gap = rng.choice([0,3,6,12,24,48,72])
        tol = rng.choice([3,6,12,24])
        res = rng.choice([10,20,30])
        t0=datetime(2024,7,rng.choice([1,5,10,15]),10,tzinfo=timezone.utc)

        arts=[
            A(name=f"flood-mask-{res}",quantity="flood_extent",scale="binary",
              crs=crs,resolution_m=res,attrs={"cloud_mask_applied":cloud}),
            A(name="S1",quantity="backscatter",acquisition_time=t0.isoformat(),
              resolution_m=10),
            A(name="S2",quantity="reflectance",
              acquisition_time=(t0+timedelta(hours=gap)).isoformat(),
              resolution_m=10)
        ]
        steps=[
            Step("require_cloud_mask",{"artifact":0}),
            Step("compare_sensors",{"a":1,"b":2,"tolerance_h":tol}),
            Step("area",{"artifact":0,"method":area_method,"output_unit":"km2"})
        ]

        if not cloud:
            violations.append("cloud mask missing")
        if gap>tol:
            violations.append("sensor timing mismatch")
        if crs=="EPSG:4326" and area_method=="planar":
            violations.append("angular planar area")

    elif template=="landslide_forecast":
        day=rng.choice([1,5,10,15,20])
        decision=datetime(2024,8,day,12,tzinfo=timezone.utc)
        delta=rng.choice([1,3,6,12,24])
        future=rng.choice([True,False])
        prov_missing=rng.choice([True,False])
        model=rng.choice(["DSM","DTM"])
        dem_res=rng.choice([30,60,90])
        rain_time = decision + timedelta(hours=delta) if future else decision-timedelta(hours=delta)

        arts=[
            A(name="IMERG-feature",quantity="rainfall",unit="mm",
              available_time=rain_time.isoformat(),
              provenance=None if prov_missing else "NASA GPM IMERG"),
            A(name=f"terrain-{dem_res}",quantity="elevation",unit="m",
              surface_model=model,vertical_datum="EGM2008",
              resolution_m=dem_res,provenance="Copernicus DEM")
        ]
        steps=[
            Step("predict",{"features":[0]}),
            Step("require_bare_earth",{"artifact":1})
        ]
        if future:
            violations.append("future feature")
        if model=="DSM":
            violations.append("DSM used for bare-earth requirement")
        if prov_missing:
            unresolved.append("missing rainfall provenance")

        intent=Intent("landslide forecast",decision_time=decision.isoformat())
        expected="REFUTED" if violations else ("UNPROVEN" if unresolved else "PROVEN")
        return BenchCase("", "blind-composition","blind",template,arts,steps,intent,
                         expected,"; ".join(violations+unresolved) or "all constraints satisfied")

    elif template=="landcover_change":
        method=rng.choice(["nearest","mode","bilinear","cubic","lanczos"])
        target=rng.choice([20,30,40,50,60,90])
        native=rng.choice([10,20])
        prov_missing=rng.choice([True,False])
        evidence = target >= native

        arts=[A(name=f"WorldCover-{native}",quantity="land_cover",
                scale="categorical",crs="EPSG:4326",resolution_m=native,
                provenance=None if prov_missing else "ESA WorldCover",
                allowed_values=[10,20,30,40,50,60,70,80,90,95,100])]
        steps=[Step("resample",{"artifact":0,"method":method,
                                "target_resolution_m":target,
                                "evidence_of_new_information":evidence})]

        if method in CONTINUOUS_INTERPOLATORS:
            violations.append("continuous categorical interpolation")
        if target<native and not evidence:
            violations.append("unsupported resolution gain")
        if prov_missing:
            unresolved.append("missing provenance")

    elif template=="flood_depth":
        terrain_datum="EGM2008"
        water_datum=rng.choice(["EGM2008","EGM96","WGS84_ellipsoid",None])
        model=rng.choice(["DSM","DTM"])
        water_unit=rng.choice(["m","km","cm"])
        convert_to=rng.choice(["m","km","km2","K"])
        dem_res=rng.choice([10,30,90])

        arts=[
            A(name=f"terrain-{dem_res}",quantity="elevation",unit="m",
              vertical_datum=terrain_datum,surface_model=model,
              resolution_m=dem_res,provenance="Copernicus DEM"),
            A(name="water-surface",quantity="water_level",unit=water_unit,
              vertical_datum=water_datum,provenance="hydraulic model")
        ]
        steps=[
            Step("combine_vertical",{"a":0,"b":1}),
            Step("convert_unit",{"artifact":1,"to":convert_to})
        ]

        if water_datum is None:
            unresolved.append("water vertical datum missing")
        elif water_datum != terrain_datum:
            violations.append("vertical datum mismatch")

        srcdim=unit_dimension(water_unit)
        dstdim=unit_dimension(convert_to)
        if srcdim is not None and dstdim is not None:
            if srcdim != dstdim and not ({srcdim,dstdim} <= {"length","length_or_accum"}):
                violations.append("dimension-changing unit conversion")

    else:  # eo_ml_validation
        mode=rng.choice(["spatial","temporal","scene"])
        arts=[A(name=f"EO-samples-{rng.randint(1,9999)}",quantity="training_samples")]

        if mode=="spatial":
            corr=rng.randint(250,5000)
            sep=rng.randint(0,7000)
            steps=[Step("split",{"mode":"spatial",
                                "separation_m":sep,
                                "dependence_range_m":corr})]
            if sep<corr:
                violations.append("spatial dependence leakage")

        elif mode=="temporal":
            test=datetime(2024,rng.choice([5,6,7,8]),1,tzinfo=timezone.utc)
            train=test+timedelta(days=rng.randint(-30,30))
            steps=[Step("split",{"mode":"temporal",
                                "train_end":train.isoformat(),
                                "test_start":test.isoformat()})]
            if train>=test:
                violations.append("temporal overlap")

        else:
            overlap=round(rng.uniform(0,0.75),4)
            steps=[Step("split",{"mode":"scene",
                                "scene_overlap_fraction":overlap})]
            if overlap>0:
                violations.append("scene overlap")

    intent=Intent(template)
    expected="REFUTED" if violations else ("UNPROVEN" if unresolved else "PROVEN")
    return BenchCase("", "blind-composition","blind",template,arts,steps,intent,
                     expected,"; ".join(violations+unresolved) or "all constraints satisfied")


def generate_balanced_blind(target_per_template=50):
    all_cases=[]
    global_seen=set()

    for j,template in enumerate(BLIND_TEMPLATES):
        rng=random.Random(SEED+900000+j*10000)
        local=[]
        attempts=0

        while len(local)<target_per_template and attempts<100000:
            attempts+=1
            c=make_blind_case(template,attempts,rng)
            sig=scientific_signature(c.artifacts,c.steps,c.intent)
            if sig in global_seen:
                continue
            global_seen.add(sig)
            c.case_id=f"SGX3_BLIND_{template}_{len(local):03d}"
            local.append(c)

        if len(local) != target_per_template:
            raise RuntimeError(
                f"Blind template {template}: only {len(local)} unique cases, "
                f"expected {target_per_template}."
            )
        all_cases.extend(local)

    return all_cases


blind_cases=generate_balanced_blind(50)

cases=single_cases+blind_cases
signatures=[scientific_signature(c.artifacts,c.steps,c.intent) for c in cases]

print("TOTAL CASES:",len(cases))
print("UNIQUE CONFIGURATIONS:",len(set(signatures)))
print("DUPLICATES:",len(signatures)-len(set(signatures)))
print("\nSplit counts")
print(pd.Series([c.split for c in cases]).value_counts())
print("\nBlind template counts")
print(pd.Series([c.family for c in blind_cases]).value_counts())


## Benchmark quality gates

These assertions prevent the notebook from quietly returning to the v2 duplication problem.

In [ ]:
assert len(signatures) == len(set(signatures)), "Duplicate scientific configurations detected."
assert len(cases) >= 400, "Benchmark is smaller than the intended v3 stress-test size."
assert sum(c.split=="blind-composition" for c in cases) >= 100
assert len(set(c.family for c in cases if c.split=="heldout-family")) >= 8
print("All benchmark quality gates passed.")

# Part IX — Baselines and evaluation

No fake LLM score is inserted.

Baselines:

- runtime-only;
- metadata-only;
- schema-lite;
- SciGeoGuard-X v3.

In [ ]:
def baseline_runtime(c):
    return "PROVEN"

def baseline_metadata(c):
    if any(x.provenance is None for x in c.artifacts):
        return "UNPROVEN"
    if any(x.vertical_datum is None for x in c.artifacts) and any(s.op=="combine_vertical" for s in c.steps):
        return "UNPROVEN"
    return "PROVEN"

def baseline_schema(c):
    # only gross structural checks, not scientific semantics
    if not c.artifacts:
        return "REFUTED"
    if any(x.resolution_m is not None and x.resolution_m<=0 for x in c.artifacts):
        return "REFUTED"
    if any(x.provenance is None for x in c.artifacts):
        return "UNPROVEN"
    return "PROVEN"

rows=[]
for c in cases:
    vr=evaluate_case(c)
    rows.append({
        "case_id":c.case_id,
        "split":c.split,
        "tier":c.tier,
        "family":c.family,
        "expected":c.expected,
        "runtime":baseline_runtime(c),
        "metadata_only":baseline_metadata(c),
        "schema_lite":baseline_schema(c),
        "scigeoguard_x_v3":vr["verdict"],
        "n_findings":len(vr["findings"]),
        "signature":scientific_signature(c.artifacts,c.steps,c.intent)
    })
results=pd.DataFrame(rows)
results.head()

In [ ]:
LABELS=["PROVEN","REFUTED","UNPROVEN"]

def classification_metrics(df, pred_col):
    y=df.expected
    p=df[pred_col]

    per_class={}
    f1s=[]
    for lab in LABELS:
        tp=((y==lab)&(p==lab)).sum()
        fp=((y!=lab)&(p==lab)).sum()
        fn=((y==lab)&(p!=lab)).sum()
        precision=tp/(tp+fp) if tp+fp else np.nan
        recall=tp/(tp+fn) if tp+fn else np.nan
        f1=2*precision*recall/(precision+recall) if precision==precision and recall==recall and precision+recall else np.nan
        per_class[lab]={"precision":precision,"recall":recall,"f1":f1}
        if f1==f1: f1s.append(f1)

    invalid=y.eq("REFUTED")
    valid=y.eq("PROVEN")

    return {
        "accuracy":float((y==p).mean()),
        "macro_f1":float(np.mean(f1s)) if f1s else np.nan,
        "FSCR":float((p[invalid]=="PROVEN").mean()) if invalid.any() else np.nan,
        "false_refutation_rate":float((p[valid]=="REFUTED").mean()) if valid.any() else np.nan,
        "proof_coverage":float(p.isin(["PROVEN","REFUTED"]).mean()),
        "valid_certification_rate":float((p[valid]=="PROVEN").mean()) if valid.any() else np.nan
    }

summary=[]
for split in ["development","heldout-family","blind-composition","all"]:
    d=results if split=="all" else results[results.split==split]
    for system in ["runtime","metadata_only","schema_lite","scigeoguard_x_v3"]:
        summary.append({"split":split,"system":system,**classification_metrics(d,system)})

summary_df=pd.DataFrame(summary)
summary_df

In [ ]:
family_perf=(
    results.assign(correct=lambda d:d.expected.eq(d.scigeoguard_x_v3))
    .groupby(["split","family"])
    .agg(
        n=("case_id","size"),
        accuracy=("correct","mean"),
        proven=("expected",lambda s:int((s=="PROVEN").sum())),
        refuted=("expected",lambda s:int((s=="REFUTED").sum())),
        unproven=("expected",lambda s:int((s=="UNPROVEN").sum()))
    )
    .reset_index()
)
family_perf

In [ ]:
confusion = pd.crosstab(
    results.expected,
    results.scigeoguard_x_v3,
    rownames=["Expected"],
    colnames=["Predicted"],
    dropna=False
).reindex(index=LABELS,columns=LABELS,fill_value=0)

confusion

# Part X — Paper-ready figures

Each figure is saved separately.

In [ ]:
import matplotlib.pyplot as plt

fig1 = RUN_DIR/"figure_fscr_by_system.png"
plotdf=summary_df[summary_df.split=="all"].copy()
plt.figure(figsize=(7,4))
plt.bar(plotdf.system,plotdf.FSCR)
plt.ylabel("False Scientific Certification Rate")
plt.xlabel("System")
plt.xticks(rotation=25,ha="right")
plt.tight_layout()
plt.savefig(fig1,dpi=200)
plt.show()

print(fig1)

In [ ]:
fig2 = RUN_DIR/"figure_proof_coverage_by_split.png"
plotdf=summary_df[summary_df.system=="scigeoguard_x_v3"].copy()
plt.figure(figsize=(7,4))
plt.bar(plotdf["split"],plotdf["proof_coverage"])
plt.ylabel("Proof coverage")
plt.xlabel("Evaluation split")
plt.ylim(0,1.05)
plt.xticks(rotation=20,ha="right")
plt.tight_layout()
plt.savefig(fig2,dpi=200)
plt.show()

print(fig2)

In [ ]:
fig3 = RUN_DIR/"figure_family_accuracy.png"
plotdf=family_perf.sort_values(["split","family"]).copy()
labels=(plotdf["split"]+" | "+plotdf["family"]).tolist()
plt.figure(figsize=(10,max(5,len(labels)*0.28)))
plt.barh(labels,plotdf["accuracy"])
plt.xlabel("SciGeoGuard-X v3 exact verdict accuracy")
plt.xlim(0,1.05)
plt.tight_layout()
plt.savefig(fig3,dpi=200)
plt.show()

print(fig3)

# Part XI — Expert-review and blind-test exports

The review sheet intentionally contains **no verifier explanation** in the expert-label columns.

In [ ]:
bench_rows=[]
review_rows=[]
jsonl_records=[]

for c in cases:
    sig=scientific_signature(c.artifacts,c.steps,c.intent)
    vr=evaluate_case(c)

    rec={
        "case_id":c.case_id,
        "split":c.split,
        "tier":c.tier,
        "family":c.family,
        "expected":c.expected,
        "truth_reason":c.truth_reason,
        "signature":sig,
        "intent":json.dumps(asdict(c.intent),default=str),
        "artifacts":json.dumps([asdict(x) for x in c.artifacts],default=str),
        "steps":json.dumps([asdict(x) for x in c.steps],default=str),
        "verifier_verdict":vr["verdict"],
        "verifier_findings":json.dumps(vr["findings"],default=str)
    }
    bench_rows.append(rec)
    jsonl_records.append(rec)

    review_rows.append({
        "case_id":c.case_id,
        "split":c.split,
        "tier":c.tier,
        "workflow_intent":c.intent.goal,
        "artifacts_json":rec["artifacts"],
        "steps_json":rec["steps"],
        "reviewer_validity":"",
        "reviewer_error_family":"",
        "reviewer_severity":"",
        "reviewer_comments":"",
        "reviewer_id":"",
        "adjudicated_validity":""
    })

bench_df=pd.DataFrame(bench_rows)
review_df=pd.DataFrame(review_rows)

print(bench_df.shape,review_df.shape)

# Part XII — Export everything and create the ZIP

This cell is intended to be the **final cell** in Colab.

In [ ]:
# ---------- summary JSON ----------
real_results = {
    "sentinel2": sentinel_result,
    "worldcover": worldcover_result,
    "imerg": imerg_result,
    "copernicus_dem": copdem_result
}

paths = {}

paths["benchmark_csv"] = RUN_DIR/"SilentGeoBench_v3.csv"
paths["benchmark_jsonl"] = RUN_DIR/"SilentGeoBench_v3.jsonl"
paths["results_csv"] = RUN_DIR/"SciGeoGuard_X_v3_results.csv"
paths["metrics_csv"] = RUN_DIR/"SciGeoGuard_X_v3_metrics.csv"
paths["family_csv"] = RUN_DIR/"SciGeoGuard_X_v3_family_performance.csv"
paths["review_csv"] = RUN_DIR/"SilentGeoBench_v3_expert_review_template.csv"
paths["real_json"] = RUN_DIR/"SciGeoGuard_X_v3_real_data_results.json"
paths["sources_json"] = RUN_DIR/"SciGeoGuard_X_v3_source_registry.json"
paths["confusion_csv"] = RUN_DIR/"SciGeoGuard_X_v3_confusion_matrix.csv"
paths["manifest"] = RUN_DIR/"MANIFEST.txt"

bench_df.to_csv(paths["benchmark_csv"],index=False)
results.to_csv(paths["results_csv"],index=False)
summary_df.to_csv(paths["metrics_csv"],index=False)
family_perf.to_csv(paths["family_csv"],index=False)
review_df.to_csv(paths["review_csv"],index=False)
confusion.to_csv(paths["confusion_csv"])

with paths["benchmark_jsonl"].open("w",encoding="utf-8") as f:
    for rec in jsonl_records:
        f.write(json.dumps(rec,default=str)+"\n")

paths["real_json"].write_text(json.dumps(real_results,indent=2,default=str),encoding="utf-8")
paths["sources_json"].write_text(json.dumps(SOURCE_REGISTRY,indent=2),encoding="utf-8")

with paths["manifest"].open("w",encoding="utf-8") as f:
    f.write("SciGeoGuard-X Publication Experiments v3\n")
    f.write("="*50+"\n\n")
    f.write(f"Total benchmark cases: {len(cases)}\n")
    f.write(f"Unique scientific configurations: {len(set(signatures))}\n")
    f.write(f"Development cases: {sum(c.split=='development' for c in cases)}\n")
    f.write(f"Held-out-family cases: {sum(c.split=='heldout-family' for c in cases)}\n")
    f.write(f"Blind-composition cases: {sum(c.split=='blind-composition' for c in cases)}\n\n")
    f.write("IMPORTANT: generated benchmark cases are not expert-reviewed until qualified reviewers annotate them.\n\n")
    f.write("Real-data mode:\n")
    f.write(json.dumps(real_results,indent=2,default=str))
    f.write("\n\nFiles:\n")
    for p in sorted(RUN_DIR.iterdir()):
        if p.is_file():
            f.write(f"- {p.name}\n")

zip_base = str(RUN_DIR.parent/"SciGeoGuard_X_v3_All_Results")
zip_path = shutil.make_archive(zip_base,"zip",root_dir=RUN_DIR)

print("Created publication bundle:")
print(zip_path)
print("\nIncluded files:")
for p in sorted(RUN_DIR.iterdir()):
    if p.is_file():
        print(" -",p.name)

# Colab automatic download.
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("\nAutomatic Colab download unavailable here.")
    print("ZIP path:",zip_path)

# Interpretation guardrails

A strong v3 result means:

- real EO access works for the chosen products;
- the verifier detects silent scientific inconsistencies across unique generated workflows;
- held-out families and blind compositions remain testable;
- `UNPROVEN` is preserved when evidence is genuinely missing.

It does **not** by itself prove universal scientific correctness.

Before manuscript submission, the remaining high-value work is:

1. freeze the verifier;
2. obtain independent expert annotations for a substantial subset;
3. add an actual LLM-judge baseline;
4. measure inter-rater agreement;
5. report blind post-freeze performance separately from generated development performance.